# SCI1070 — Semaine 13 : quatre mécanismes, isolés et mesurés

| Volet | Ce que vous y faites | Durée |
| --- | --- | --- |
| A | les trois estimateurs de retour, **comme fonctions pures** | dix minutes |
| B | la semaine 3, sur une dynamique **analytique** | quelques minutes |
| C | un gradient qui traverse **deux couches de dynamique** | vingt lignes |
| D | **une politique qui désapprend**, puis qui cesse de désapprendre | le plus important |

**Rien de ce que cette semaine décrit n'est reproductible, et je ne vais pas faire
semblant.** Le laboratoire **isole quatre mécanismes** et les mesure séparément.

> **Aucun des quatre volets n'a été exécuté par ce cours.** Les volets sont écrits
> comme des **consignes**, jamais comme des résultats attendus. **Si votre résultat
> diffère, dites ce que vous avez obtenu et expliquez ce que cela mesure.**

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium[classic-control]>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)
if importlib.util.find_spec("sci1070") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "git+https://github.com/FaycalZM/drl-course-teluq.git#subdirectory=code"],
                   check=True)

import collections
import copy

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sci1070.protocole import intervalle
from sci1070.courbes import tracer_bande
from sci1070.graines import fixer_graines

device = torch.device("cpu")
GRAINES = [0, 1, 2]
GAMMA = 0.99

---
# Volet A : les trois estimateurs, comme fonctions pures

In [ ]:
def V_R(recompenses, valeurs, gamma=GAMMA):
    """L'estimateur sans amorçage. Ne doit JAMAIS lire `valeurs`."""
    ...        # À COMPLÉTER : la somme actualisée des récompenses, et rien d'autre


def V_N(recompenses, valeurs, n, gamma=GAMMA):
    """L'estimateur à n pas."""
    ...        # À COMPLÉTER : n récompenses, puis la valeur amorcée


def V_lambda(recompenses, valeurs, lam, gamma=GAMMA):
    """La moyenne pondérée. Une récurrence, et elle se décale facilement d'un indice."""
    ...        # À COMPLÉTER : la récurrence arrière. Les deux cas limites lam=0 et lam=1 la vérifient.

### Les trois identités

In [ ]:
rng = np.random.default_rng(0)
r = torch.as_tensor(rng.normal(size=40), dtype=torch.float32)
v = torch.as_tensor(rng.normal(size=40), dtype=torch.float32)

# 1. V_N avec n = 1 doit valoir la cible à un pas de la semaine 5.
cible_un_pas = r[:-1] + GAMMA * v[1:]
assert torch.allclose(V_N(r, v, n=1)[:-1], cible_un_pas, atol=1e-4)

# 2. V_lambda(0) = V_N(1), et V_lambda(1) = V_N(H).
assert torch.allclose(V_lambda(r, v, lam=0.0), V_N(r, v, n=1), atol=1e-4)
assert torch.allclose(V_lambda(r, v, lam=1.0), V_N(r, v, n=len(r)), atol=1e-4)

# 3. V_R ne doit JAMAIS dépendre du tenseur de valeurs.
assert torch.allclose(V_R(r, v), V_R(r, torch.full_like(v, 1000.0)))
print("les trois identités passent")

### Étape 2 — l'expérience qui porte l'argument de la section 13.11

In [ ]:
T, PAS_RECOMPENSE = 40, 30
recompenses = torch.zeros(T)
recompenses[PAS_RECOMPENSE] = 1.0

# Le vrai retour actualisé restant, depuis chaque pas. Calculable à la main.
valeurs_parfaites = torch.tensor(
    [GAMMA ** (PAS_RECOMPENSE - t) if t <= PAS_RECOMPENSE else 0.0 for t in range(T)],
    dtype=torch.float32)

print("récompense au pas", PAS_RECOMPENSE)
print("valeur parfaite au pas 0 :", float(valeurs_parfaites[0]))

In [ ]:
...        # À COMPLÉTER : les trois courbes en fonction de l'horizon H, de dix à quarante
ax.axvline(PAS_RECOMPENSE, linestyle="--", linewidth=1,
           label=f"la récompense arrive au pas {PAS_RECOMPENSE}")
ax.set_xlabel("horizon H"); ax.set_ylabel("estimation au pas initial")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Ce que vous devriez voir** : \(V_{\mathrm{R}}\) reste **exactement nul** jusqu'à
\(H = 30\) puis se réveille brutalement, pendant que les deux autres donnent une
estimation correcte **dès \(H = 10\)**.

**Étape 3, à écrire dans votre carnet.** Elle établit que \(V_{\mathrm{R}}\) est
aveugle au-delà de l'horizon, et que l'amorçage lève cette cécité. Elle **n'établit
pas** que l'agent complet apprend mieux : **votre critique était parfait, et un
critique parfait est précisément ce qu'on n'a pas.**

---
# Volet B : la semaine 3, sur une dynamique analytique

In [ ]:
class DoubleIntegrateur:
    """Une masse ponctuelle sur une ligne. État (x, xdot), action = accélération."""

    def __init__(self, dt=0.05, borne=1.0):
        self.dt, self.borne = dt, borne

    def pas(self, etat, action):
        """Un pas d'Euler. Exact, réinitialisable, gratuit."""
        x, xdot = float(etat[0]), float(etat[1])
        u = float(np.clip(np.atleast_1d(action)[0], -self.borne, self.borne))
        xdot_s = xdot + u * self.dt
        x_s = x + xdot_s * self.dt
        return np.array([x_s, xdot_s]), -(x_s ** 2) - 0.1 * (xdot_s ** 2)

    def derouler(self, etat, actions):
        total = 0.0
        for u in actions:
            etat, r = self.pas(etat, np.atleast_1d(u))
            total += r
        return total


dyn = DoubleIntegrateur()
print("un pas depuis (1, 0) avec u = -1 :", dyn.pas(np.array([1.0, 0.0]), [-1.0]))

In [ ]:
def planifier(dynamique, etat, H=12, I=10, J=1000, K=100, rng=None,
              reinitialiser=True, loi_precedente=None):
    """La fonction de la section 13.5, les appels au modèle remplacés par la dynamique."""
    ...        # À COMPLÉTER : c'est votre boucle de la semaine 3 : tirer, garder les K meilleures, resserrer, recommencer I fois, et ne jouer que la première action

### Étape 3 — les trois mesures qui font le volet

In [ ]:
...        # À COMPLÉTER : la réinitialisation de la loi, et la perturbation brusque au pas cent

In [ ]:
...        # À COMPLÉTER : les itérations contre le tirage brut, à budget de candidats ÉGAL

In [ ]:
...        # À COMPLÉTER : l'horizon, de 3 à 30

**Étape 4, ce que le volet n'établit pas** : **rien de ceci ne dit quoi que ce soit
sur la qualité d'un modèle appris.** Votre dynamique est exacte. Vous avez mesuré le
**planificateur**, et uniquement lui.

---
# Volet C : un gradient qui traverse deux couches de dynamique

In [ ]:
def geler(module):
    for p in module.parameters():
        p.requires_grad_(False)
    return module


torch.manual_seed(0)
T1 = geler(nn.Sequential(nn.Linear(3, 16), nn.Tanh(), nn.Linear(16, 2)))
T2 = geler(nn.Sequential(nn.Linear(3, 16), nn.Tanh(), nn.Linear(16, 2)))
tete_recompense = geler(nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 1)))
print("trois modules gelés")

In [ ]:
class ActeurReparametre(nn.Module):
    """Celui de la section 13.9 : tanh, et le bruit tiré À L'EXTÉRIEUR."""

    def __init__(self, n_etat, n_action):
        super().__init__()
        self.pipe = nn.Sequential(nn.Linear(n_etat, 16), nn.Tanh(),
                                  nn.Linear(16, 2 * n_action))
        self.n_action = n_action

    def forward(self, etat, bruit, reparametrer=True):
        ...        # À COMPLÉTER : l'action est une FONCTION du bruit, pas un tirage

In [ ]:
def norme_du_gradient_acteur(detacher_apres_t1=False, sans_reparametrisation=False):
    """Déroule deux pas, lit la récompense, rétropropage, et rend la norme."""
    ...        # À COMPLÉTER : état initial -> acteur -> T1 -> acteur -> T2 -> récompense


n_correct = norme_du_gradient_acteur()
n_sans_repar = norme_du_gradient_acteur(sans_reparametrisation=True)
n_detache = norme_du_gradient_acteur(detacher_apres_t1=True)

print(f"correct            : {n_correct:.6f}")
print(f"sans reparamétrage : {n_sans_repar:.6f}   <- le gradient n'existe plus")
print(f"détaché après T1   : {n_detache:.6f}   <- non nul, mais amputé")
assert n_sans_repar < n_correct / 10, "le bogue de la section 13.9 ne se voit pas"
assert 0 < n_detache < n_correct
print()
print("Mesure, et elle est plus tranchée que l'énoncé ne le laisse attendre :")
print("sans reparamétrisation le gradient ne s'effondre pas, il DISPARAÎT, et")
print("PyTorch lève au lieu de rendre zéro. Dans un vrai entraînement, l'erreur")
print("est donc bruyante ici ; elle devient silencieuse dès qu'une autre")
print("branche du graphe garde l'objectif dérivable.")

**Étape 4, la comparaison qui vaut la peine.**

In [ ]:
...        # À COMPLÉTER : un acteur de la semaine 9 : il maximise une valeur d'action AU PAS COURANT, sans transition dans le chemin

**Ce que ce volet n'établit pas** : il ne montre **pas** que ce gradient entraîne un
bon agent. **Un chemin de dérivation peut exister et être totalement inutile, si le
modèle qu'il traverse est faux.**

---
# Volet D : une politique qui désapprend, puis qui cesse de désapprendre

**Étape 1 — fabriquer le jeu figé, et c'est la seule fois où vous appelez Gymnasium.**

In [ ]:
def politique_comportement(obs, rng, gain=2.0, sigma=1.5):
    """Médiocre mais pas absurde : proportionnel grossier, plus bruit généreux."""
    ...        # À COMPLÉTER : un contrôleur proportionnel sur l'angle, et du bruit


def collecter_jeu(n_transitions, graine=0, chemin=None):
    """Appelle env.step. C'est la SEULE fonction du volet qui en a le droit."""
    ...        # À COMPLÉTER : collecter, puis np.savez sur disque


for n in [1_000, 10_000, 50_000]:
    collecter_jeu(n, chemin=f"jeu-{n}.npz")
print("trois jeux figés écrits sur disque. À partir d'ici, plus aucun env.step.")

**Étape 2 — désapprendre.** Trois quantités, et **c'est le choix de ces trois
quantités qui fait le volet**.

In [ ]:
def entrainer_hors_ligne(chemin_jeu, pas_gradient=3000, alpha_cons=0.0, graine=0,
                         mesurer_tous=100):
    """Rend (retours_reels, valeurs_estimees, erreurs_bellman)."""
    ...        # À COMPLÉTER : charger le jeu DEPUIS LE FICHIER, et ne jamais collecter

In [ ]:
...        # À COMPLÉTER : les trois courbes sur le même axe des abscisses

**Étape 3, l'expérience qui doit vous marquer.**

In [ ]:
...        # À COMPLÉTER : les trois tailles, sur trois graines, avec intervalle

**Étape 4, cesser de désapprendre.**

In [ ]:
...        # À COMPLÉTER : trois valeurs d'alpha_cons séparées par un facteur dix

**Étape 5, la référence qu'il faut absolument tracer.**

In [ ]:
...        # À COMPLÉTER : le retour de la politique de comportement elle-même

---
# La solution courte de cette semaine est un clonage de comportement

In [ ]:
# La reference la plus honnete pour un apprentissage hors ligne : un
# apprentissage supervise pur sur le jeu figé. Aucune recompense, aucune valeur,
# aucun amorçage, donc AUCUN des problemes de la section 13.14.

...        # À COMPLÉTER : le clonage de comportement, sur le MÊME jeu figé

> **Pourquoi cette référence est meilleure qu'une bibliothèque, dans ce cas précis.**
> Une référence de bibliothèque vous dit si votre **code** est correct. Celle-ci vous
> dit **si votre problème valait la peine d'être posé.** Un jeu de données sur lequel
> un clonage de comportement égale votre meilleure méthode hors ligne est un jeu qui
> ne contient **aucune recomposition exploitable**, et aucune sophistication n'y
> changera rien.